# 01 · Pairs: measure the population, freeze the split, build the honest test

*For learning. Nothing here runs in CI and nothing leaves this computer: the course content, the local MLflow server
(`http://localhost:5050`, experiment `coach-retrieval-lab`) and Ollama for one local model.*

**What this notebook settles.** Before any model is scored, three things have to be nailed down, once, and never moved
again: **what the test questions are** (the pairs), **which lessons each stage may look at** (the split), and **a test
that is fair** (questions in a learner's own words, not the course's). Everything is registered in MLflow in one run,
`pairs · build and freeze (seed 7)`, with a digest (a fingerprint of the rows' text and number columns) per dataset, so stage 2 onwards
can say precisely which data it saw.

**Which split it uses.** It *makes* the split. It shows the **train**, **dev** and **test** lessons. The **holdout**
lessons are registered too, but their rows are never printed here and no notebook may score on them until the very
end of the lab.

**The champion, said plainly.** Today's coach does *not* retrieve with embeddings: the page tells it which lesson the
learner is on. So for the retrieval step the champion is `bge-large` exactly as its makers ship it (recall@5 0.84 on
150 Check questions, PR #170), and a fine-tune has to earn a place where the lesson is *not* known: a question typed on
the study list, to the blueprint's coach, or through Claude's tools without a lesson. No real typed learner questions are
recorded on this computer yet (0 events), which is why section 4 has to make learner-worded questions.

**How to work through it.** Before each result there is a cell for **your guess**: write it down, then run the next cell.
Your guesses go into the MLflow run beside the real answers. The exercises have blanks (`"___"`) checked by an assert;
the answers are in `01_pairs_answers.py` if you are stuck.

> **In plain words: a seed.** Anything that uses chance, like shuffling lessons or picking a sample, needs a starting
> number to begin its "random" choices from. That number is the **seed** (the real term: random seed). The same seed
> gives the same shuffle every time, on any computer, so a split or a sample can be written down and repeated; a
> different seed is a fresh roll of the dice. This lab fixes seed 7 for the split and the samples so they never move by
> accident, and stage 3 trains with two different seeds on purpose, to see how much luck alone moves a score.


In [1]:
import importlib
import json
import os
import time
from collections import Counter

os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import mlflow
import pandas as pd
from drill.content import build_data

import retrieval as r
import tracking

FILL = os.environ.get("DRILL_FILL_ANSWERS") == "1"      # only for checking the notebook end to end
answers = importlib.import_module("01_pairs_answers")
pd.set_option("display.max_colwidth", 110)


def attempt(value, name):
    # Your answer; if it still has a blank, a reminder (or, when checking the notebook end to end, the answer file's).
    if "___" not in str(value):
        return value
    if FILL:
        return getattr(answers, name)
    print("Fill in the ___ first, then run this cell again.")
    return None


data = build_data()
tracking.connect()
run = mlflow.start_run(run_name=f"pairs · build and freeze (seed {r.SEED})",
                       description="(the full description is written at the end of the notebook, once the counts are known)")
mlflow.set_tags({"kind": "coach_retrieval_lab", "lab.stage": "01_pairs", "git_commit": tracking.git_commit(),
                 **{f"computer.{k}": str(v) for k, v in tracking.this_computer().items()}})
tracking.guard_run()   # a cell that raises ends the run as FAILED, not as an empty FINISHED shell
you = tracking.Guesses()
print(f"MLflow run {tracking.short(run.info.run_id)} in experiment {run.info.experiment_id}; answers filled in: {FILL}")

MLflow run b5d212a7 in experiment 6; answers filled in: True


## 1. What a pair is, and how many (measure the population first)

> **In plain words.** The coach's retrieval step is a search: the learner's question is the needle, every Learn
> paragraph of the course is the **haystack** (the real term is the **corpus**). A **pair** is one question together with
> the lesson whose paragraphs answer it. The pairs are both the test (does the right lesson come up among the five
> closest paragraphs? that is **recall@5**, the course's "Measuring retrieval") and, later, the training data.
>
> Measure the whole population before sampling anything: a sample can only be called representative of something you
> have counted. So first: how many paragraphs, how many questions, from where, and in which modules.

Three kinds of question exist in the content, each written by a different hand:

* **check**: the lesson's own Check question (one per lesson);
* **more**: its extra practice questions;
* **prepared**: the coach's prepared question chips, the ones a learner taps beside a question ("what does *baseline*
  mean here?", "why not *a timeout*?").

**Your guess first.** 211 lessons, 741 paragraphs. How many pairs (questions with one known right lesson) do you expect?

> **In plain words: a chunk.** RAG cuts long documents into **chunks**, small pieces that can be searched and put in a prompt
> (*Chunking documents*). The course needs no cutting: each Learn paragraph is already short, so here one paragraph is one chunk and
> the corpus is 744 chunks. When the README says "first fix chunking", it means checking that choice before blaming the model.

In [2]:
you.guess("pairs", None)   # replace None with your guess, e.g. 900

No guess written down. The habit only works if you commit to one before you look.


In [3]:
pairs, corpus = r.build_pairs(data), r.build_corpus(data)
module_title = {m["id"]: m["title"] for m in data["modules"]}
you.reveal("pairs", len(pairs), as_percent=False)
by_source = Counter(p["source"] for p in pairs)
print(f"{len(corpus)} Learn paragraphs in {len({c['lesson'] for c in corpus})} lessons and {len(data['modules'])} modules; "
      f"{len(pairs)} pairs: {dict(by_source)}")
pd.DataFrame([{"source": s, "pairs": n, "share": f"{n / len(pairs):.0%}"} for s, n in by_source.most_common()])

pairs: it was 1,680 (no guess written down)
744 Learn paragraphs in 212 lessons and 28 modules; 1680 pairs: {'check': 212, 'more': 418, 'prepared': 1050}


,source,pairs,share
0,prepared,1050,62%
1,more,418,25%
2,check,212,13%


**Reading it.** The prepared chips are the biggest source by far, and they were not written as search queries. Keep that
in mind: a number over "all pairs" is mostly a number over chips.

**The first surprise, and what it changed.** Some chips say the same words under many lessons: "when does this stop
working?" and "how would I spot it going wrong?" sit under about 150 lessons each. On its own, such a question has no
single right lesson, so it is not a retrieval test at all. Keeping the first copy would have made *wrong* pairs (the
question paired with whichever lesson came first). So the rule became: a question that appears under two or more
different lessons is dropped entirely. Here is what was dropped:

In [4]:
ambiguous = r.ambiguous_questions(data)
dropped_copies = sum(len(a["lessons"]) for a in ambiguous)
print(f"{len(ambiguous)} question texts dropped, {dropped_copies} copies in all")
pd.DataFrame([{"question": a["question"], "under how many lessons": len(a["lessons"])} for a in ambiguous])

10 question texts dropped, 329 copies in all


,question,under how many lessons
0,when does this stop working?,151
1,how would i spot it going wrong?,151
2,why does this matter?,10
3,what does “context window” mean here?,3
4,why not “a rate limit”?,3
5,what does “state” mean here?,3
6,what does “baseline” mean here?,2
7,why not “a timeout”?,2
8,how is “embedding” different from “vector”?,2
9,how is “design” different from “evaluate”?,2


### Exercise 1: the dropping rule

"Why does this matter?" appears under 10 different lessons. How many pairs does it make?

In [5]:
PAIRS_FROM_SHARED_QUESTION = "___"   # a whole number

In [6]:
mine = attempt(PAIRS_FROM_SHARED_QUESTION, "PAIRS_FROM_SHARED_QUESTION")
if mine is not None:
    assert int(mine) == 0, "Not quite: a question under several lessons has no single right lesson, so it cannot be a pair for any of them."
    print("Right: none. It only makes sense on the page that shows it, where the lesson is already known.")

Right: none. It only makes sense on the page that shows it, where the lesson is already known.


### Hunch, then finding: the option chips

Among the prepared chips, the **option** chips ("why not *a timeout*?") quote a wrong answer and name no topic. The hunch
was that they would be hard to retrieve. It was checked on all 328 dev questions in the smoke run of `retrieval.py`
(MLflow run `smoke · retrieval.py · dev by question source`): bge-large finds the right lesson for Check questions 0.88
of the time (42 questions, range 0.75 to 0.95), for more questions 0.84 (82, 0.75 to 0.90), for term chips 0.81 (26, about
0.62 to 0.91) and for option chips **0.45** (174, 0.38 to 0.53). The ranges are Wilson 95% ranges: 26 questions say much
less than 174, which is why the term chips' range is nearly 30 points wide. So that is now a **finding**:
the option chips are a different population. *This is wrong if* learners actually type those words to the coach; but
today nobody types them, they are tapped beside the question they belong to. Stage 2 will score them as their own
group, or leave them out and say why. The counts, by chip kind and by module:

In [7]:
chips = Counter(p["chip"] for p in pairs if p["source"] == "prepared")
print("prepared chips by kind:", dict(chips.most_common()))
rows = []
for m in data["modules"]:
    mine_ = [p for p in pairs if p["module"] == m["id"]]
    rows.append({"module": m["id"], "lessons": sum(1 for l in m["lessons"] if l in data["lessons"]),
                 "paragraphs": sum(c["module"] == m["id"] for c in corpus), "pairs": len(mine_),
                 **{s: sum(p["source"] == s for p in mine_) for s in ("check", "more", "prepared")},
                 "option chips": sum(p["chip"] == "option" for p in mine_)})
population = pd.DataFrame(rows)
mlflow.log_metrics({"n_pairs": len(pairs), "n_paragraphs": len(corpus), "n_lessons": len(data["lessons"]),
                    "n_modules": len(data["modules"]), "n_ambiguous_questions": len(ambiguous),
                    **{f"n_pairs_{s}": n for s, n in by_source.items()}, "n_option_chips": chips.get("option", 0)})
population

prepared chips by kind: {'option': 888, 'term': 129, 'explain:gotcha': 33}


,module,lessons,paragraphs,pairs,check,more,prepared,option chips
0,use-case,7,34,68,7,14,47,41
1,foundations,6,20,42,6,6,30,22
2,pricing,13,54,88,13,26,49,42
3,experiments,18,69,179,18,36,125,108
4,llm-apps,6,13,39,6,12,21,18
5,rag,16,63,146,16,32,98,84
6,agents,14,42,135,14,28,93,75
7,agent-teams,8,32,55,8,16,31,24
8,databricks-ai,5,29,22,5,10,7,6
9,production,9,18,74,9,18,47,40


**What to look out for.** Three modules have fewer than four lessons (`platform-repos`, `azure-databricks`,
`troubleshooting`). Any per-module number for them will rest on a handful of questions, so the "weakest module" in a
later table can be noise before it is a fact; read the count beside it.

## 2. The split, by whole lesson ("Splits that don't move", "Dev, test and holdout sets")

> **In plain words.** The lessons are dealt into four piles. **Train** is what the fine-tune learns from. **Dev** is where
> it is watched while choices are made (how long to train, which setting). **Test** is scored only when a stage says it is
> done, to report a number nothing was tuned on. **Holdout** is scored **once, at the very end of the lab**, so the final
> number cannot have been tuned on even by accident. Shares 60/20/10/10.
>
> **Why by whole lesson, not by question.** A lesson's questions are all answered by the *same* paragraphs. If one of its
> questions trains the model and another tests it, the model is rewarded in dev for paragraphs it was pulled towards in
> training. That is **leakage**: the test contains the answers. **Why within each module.** A module is a topic. Dealing
> inside each module (the real term is **stratifying**) keeps every topic in every pile, so a dev or test score speaks for
> the whole course and the weakest module can be read.

### The leak a per-question split would cause, worked small

Take the lesson *Is it fit for purpose?*. It has three questions and four paragraphs. Deal by question and two of them
can land in different piles while sharing every paragraph:

In [8]:
lesson = "fit_for_purpose"
qs = [p for p in pairs if p["lesson"] == lesson and p["source"] in ("check", "more")]
paras = [c["id"] for c in corpus if c["lesson"] == lesson]
print(f"{data['lessons'][lesson]['title']!r}: {len(qs)} questions, paragraphs {paras}\n")
print(f"  train <- {qs[0]['id']}: {qs[0]['question'][:95]}...")
print(f"  dev   <- {qs[1]['id']}: {qs[1]['question'][:95]}...")
print(f"\nBoth are answered by {paras}. Training pulls those paragraphs towards the first question; "
      f"dev then rewards the model for finding them for the second. The dev score would be partly a memory test.")

'Is it fit for purpose?': 3 questions, paragraphs ['fit_for_purpose#0', 'fit_for_purpose#1', 'fit_for_purpose#2', 'fit_for_purpose#3']

  train <- fit_for_purpose?0: The pilot scores 86 in 100 against a criterion of 80, passes every must-pass case and costs les...
  dev   <- fit_for_purpose?1: The pilot gets 92 in 100 right, but fails one must-pass case: it told a customer to ignore a sa...

Both are answered by ['fit_for_purpose#0', 'fit_for_purpose#1', 'fit_for_purpose#2', 'fit_for_purpose#3']. Training pulls those paragraphs towards the first question; dev then rewards the model for finding them for the second. The dev score would be partly a memory test.


How common would that be? Deal the questions (not the lessons) 60/20/10/10 in the same seeded order, and count the dev
questions whose lesson also has a question in train:

> **In plain words: a hash.** A **hash** turns any text into a fixed scrambled code: here `sha256` of `"7:<question id>"` gives 64
> letters and digits. The same text always gives the same code, and a one-letter change gives a completely different one, so
> sorting by the code is a shuffle anyone can repeat on any computer (*Splits that don't move*).

In [9]:
import hashlib

order = sorted(pairs, key=lambda p: hashlib.sha256(f"{r.SEED}:{p['id']}".encode()).hexdigest())
labels = [s for s in r.SPLITS for _ in range(int(r.SHARES[s] * len(order)))]
labels += ["train"] * (len(order) - len(labels))
by_question = {p["id"]: s for p, s in zip(order, labels)}
trained_lessons = {p["lesson"] for p in pairs if by_question[p["id"]] == "train"}
dev_q = [p for p in pairs if by_question[p["id"]] == "dev"]
leaked = sum(p["lesson"] in trained_lessons for p in dev_q)
mlflow.log_metric("per_question_split_leak_share", leaked / len(dev_q))
print(f"A per-question split: {leaked} of {len(dev_q)} dev questions ({leaked / len(dev_q):.0%}) ask about a lesson the model "
      f"would have trained on. Nearly the whole dev set would be contaminated.")

A per-question split: 326 of 336 dev questions (97%) ask about a lesson the model would have trained on. Nearly the whole dev set would be contaminated.


### Freeze it

`freeze_splits` deals once, writes `data/splits.json` (seed, time, the rule, the counts and the map) and afterwards
**refuses to change the file**: if a fresh deal differs (new seed, new lessons), it raises instead of overwriting, because
every score logged so far was on the frozen split. The order is a seeded hash of each lesson's id, so the same seed deals
the same way on every computer. It does **not** survive a content change: a lesson added, removed or renamed shifts the
order and the counts, so other lessons move too (checked: one new lesson in a seven-lesson module moved three existing
ones, one of them out of holdout). That is why the file refuses by accident. A redeal is a decision, made with `force=True`,
which prints every lesson that moved, and it makes every earlier score non-comparable.

In [10]:
# Frozen means frozen: the same seed deals the same split as long as the course is the same. When a lesson has been
# added or removed since the freeze, the deal CAN move other lessons (a new lesson takes a place in its module's seeded
# order), so the only honest move is a redeal on purpose, logged, with every scores-so-far marked as "on the old split".
try:
    splits = r.freeze_splits(data, r.SEED)
    mlflow.set_tag("splits_redealt", "no")
except RuntimeError as changed:
    before = json.loads((r.DATA / "splits.json").read_text())["splits"]
    print("The course changed since the split was frozen, so this pass redeals it on purpose:\n  ", str(changed)[:160])
    splits = r.freeze_splits(data, r.SEED, force=True)
    moved = {lid: f"{before.get(lid, 'new')} -> {splits.get(lid, 'gone')}" for lid in sorted(set(before) | set(splits)) if before.get(lid) != splits.get(lid)}
    print(f"  {len(moved)} lesson(s) moved: {moved}\n  Stage 2's numbers so far were on the old split: rerun it before comparing anything to them.")
    mlflow.set_tag("splits_redealt", "yes: the course changed since the freeze")
    mlflow.log_param("lessons_moved_on_redeal", json.dumps(moved)[:500])
frozen = json.loads((r.DATA / "splits.json").read_text())
counts = r.split_counts(splits)
pairs_by_split = Counter(splits[p["lesson"]] for p in pairs)
print(f"frozen at {frozen['frozen_at']} with seed {frozen['seed']}\nrule: {frozen['rule']}\n")
print(f"lessons per split: {counts}\npairs per split:   {dict((s, pairs_by_split[s]) for s in r.SPLITS)}")
mlflow.log_params({"seed": r.SEED, "split_rule": frozen["rule"], "split_shares": "60/20/10/10 train/dev/test/holdout"})
mlflow.log_metrics({**{f"lessons_{s}": n for s, n in counts.items()}, **{f"pairs_{s}": pairs_by_split[s] for s in r.SPLITS}})
rows = []
for m in data["modules"]:
    ls = [l for l in m["lessons"] if l in data["lessons"]]
    c = Counter(splits[l] for l in ls)
    rows.append({"module": m["id"], "lessons": len(ls), **{s: c.get(s, 0) for s in r.SPLITS}})
pd.DataFrame(rows)

The course changed since the split was frozen, so this pass redeals it on purpose:
   /Users/huydo/Developer/dd-retrieval/notebooks/learning/coach_retrieval/data/splits.json is frozen and differs from a fresh deal (seed 7). Scores so far used the
REDEAL on purpose (force=True): 2 lessons change split; every score logged before this is no longer comparable.
  jury_vote: (new) -> dev
  weighted_score: dev -> test
  2 lesson(s) moved: {'jury_vote': 'new -> dev', 'weighted_score': 'dev -> test'}
  Stage 2's numbers so far were on the old split: rerun it before comparing anything to them.
frozen at 2026-11-08T17:41:06+00:00 with seed 7
rule: by whole lesson, within each module, seeded hash order, 60/20/10/10; modules with 4+ lessons have at least one in train, dev and test; smaller modules fill train, dev, train

lessons per split: {'train': 123, 'dev': 42, 'test': 29, 'holdout': 18}
pairs per split:   {'train': 970, 'dev': 321, 'test': 236, 'holdout': 153}


,module,lessons,train,dev,test,holdout
0,use-case,7,4,1,1,1
1,foundations,6,3,1,1,1
2,pricing,13,8,3,1,1
3,experiments,18,11,3,2,2
4,llm-apps,6,3,1,1,1
5,rag,16,9,3,2,2
6,agents,14,9,3,1,1
7,agent-teams,8,5,1,1,1
8,databricks-ai,5,3,1,1,0
9,production,9,5,2,1,1


**Reading it.** Every module with four or more lessons has at least one lesson in train, dev and test, so every topic is
measured. Modules with fewer than four fill train, then dev, then train again: training data is what the fine-tune needs
most, and dev is where it is watched. Only modules with seven or more lessons give a lesson to holdout (18 lessons from 13
modules).

**Frozen means frozen.** Ask for a different deal and the file refuses:

In [11]:
try:
    r.freeze_splits(data, seed=8)
except RuntimeError as e:
    print("Refused, as it should be:", str(e)[:150], "...")

Refused, as it should be: /Users/huydo/Developer/dd-retrieval/notebooks/learning/coach_retrieval/data/splits.json is frozen and differs from a fresh deal (seed 8). Scores so fa ...


### Exercise 2: where a sibling lands

`agent_scorecard?0`, the Check question of *Evaluating agents*, is in **test**. Its first extra question is
`agent_scorecard?1`. Which split must it be in?

In [12]:
SPLIT_OF_SIBLING = "___"   # "train", "dev", "test" or "holdout"

In [13]:
mine = attempt(SPLIT_OF_SIBLING, "SPLIT_OF_SIBLING")
if mine is not None:
    assert mine == splits["agent_scorecard"] == "test", "Not quite: the deal is by whole lesson, so every question of a lesson sits where the lesson sits."
    print("Right: test, like every other question of that lesson. That is the whole point of dealing by lesson.")

Right: test, like every other question of that lesson. That is the whole point of dealing by lesson.


### Exercise 3: the locked pile

Over the whole lab, from here to the final decision, how many times is the holdout split scored?

In [14]:
HOLDOUT_SCORED_TIMES = "___"   # a whole number

In [15]:
mine = attempt(HOLDOUT_SCORED_TIMES, "HOLDOUT_SCORED_TIMES")
if mine is not None:
    assert int(mine) == 1, "Not quite: every look at holdout before the end makes the final number a little less honest. It is scored exactly once."
    print("Right: once, at the very end. Every look before that makes the final number a little less honest, "
          "because what you saw could steer a choice. The code enforces it on every path (pairs_in here; score and "
          "evaluate_setup read the frozen file themselves, so leaving out the splits argument does not unlock it):")
    try:
        r.pairs_in(pairs, splits, "holdout")
    except PermissionError as e:
        print("  PermissionError:", e)

Right: once, at the very end. Every look before that makes the final number a little less honest, because what you saw could steer a choice. The code enforces it on every path (pairs_in here; score and evaluate_setup read the frozen file themselves, so leaving out the splits argument does not unlock it):
  PermissionError: the holdout split is locked: pass yes_holdout=True only for the final, once-only score


## 3. The leakage checks (leakage, from "Splits that don't move")

Before anything is scored on a split, four checks have to pass. Each is printed with what it means. A failed check is a
*stop and look*, not a number to explain away.

> **In plain words: the near-duplicate check.** Each paragraph is turned into a TF-IDF word table (the word-count rule from
> `00_vet.ipynb`, with the small common words, the **stop words**, dropped). Two paragraphs are then compared by **cosine
> similarity**: how nearly their lists of numbers point the same way, 1.0 the same, 0 no words in common (*What an embedding is:
> meaning as a place*). At 0.9 or above, two paragraphs use almost the same words in almost the same proportions: a near-copy that
> would carry an answer across the split. Why 0.9: a copy of a paragraph with one word in ten deleted still scores about 0.97
> (98 of 100 sampled copies stay at 0.9 or above), and one with one word in ten swapped for another course word scores about
> 0.88, while the two closest different paragraphs across the splits (both about the catalog, schema and service names)
> score 0.67. So 0.9 catches copies that are nearly word for word; a lightly reworded copy can sit just under it, but no real
> pair comes near it here, so any line from 0.7 to 0.9 would give the same answer.

In [16]:
MEANING = {
    "every lesson is in exactly one split": "the whole-lesson rule holds: no lesson's paragraphs sit in two piles",
    "no question appears verbatim in another split's paragraphs": "no test question is a copied sentence from a training paragraph (that would be memory, not retrieval)",
    "near-duplicate paragraphs across splits": "no two paragraphs in different piles say almost the same thing in almost the same words (recaps do this on purpose; a few would be listed with their lessons so a lesson could be moved)",
    "holdout is locked": "scoring on holdout needs yes_holdout=True and happens once, at the end",
}
checks = r.leakage_checks(pairs, corpus, splits)
for c in checks:
    meaning = next(v for k, v in MEANING.items() if c["check"].startswith(k))
    print(f"{'passed' if c['passed'] else 'FAILED'} · {c['check']}\n        what it means: {meaning}\n        detail: {c['detail'][:170]}\n")
mlflow.log_metric("leakage_checks_passed", sum(c["passed"] for c in checks))
mlflow.log_dict(checks, "leakage_checks.json")
print(f"{sum(c['passed'] for c in checks)} of {len(checks)} checks passed")

passed · every lesson is in exactly one split
        what it means: the whole-lesson rule holds: no lesson's paragraphs sit in two piles
        detail: 212 lessons with questions or paragraphs; counts {'train': 123, 'dev': 42, 'test': 29, 'holdout': 18}

passed · no question appears verbatim in another split's paragraphs
        what it means: no test question is a copied sentence from a training paragraph (that would be memory, not retrieval)
        detail: 1680 questions checked against 744 paragraphs

passed · near-duplicate paragraphs across splits (TF-IDF cosine >= 0.9)
        what it means: no two paragraphs in different piles say almost the same thing in almost the same words (recaps do this on purpose; a few would be listed with their lessons so a lesson could be moved)
        detail: 0 pairs (none)

passed · holdout is locked
        what it means: scoring on holdout needs yes_holdout=True and happens once, at the end
        detail: 18 lessons, 153 questions; scoring on 

## 4. The honest test: questions in a learner's own words ("Embedding models have blind spots")

> **In plain words.** The Check questions were written *from* the Learn text, so they borrow its words. A search that
> only matches words does well on them, and an embedding model gets credit for word overlap, not understanding. That is
> the known flaw in the easy test, and the reason the smoke run saw a word-count table (TF-IDF) keep up with bge-large.
> A learner types in their own words. Until real typed questions are recorded on this computer, the honest test is made
> by asking a local language model with about 14 billion learned numbers (`qwen3:14b`, through Ollama) to rewrite each dev and test question the way a junior
> developer would type it to a coach.
>
> "Honest test" and "easy test" are this lab's own names, not industry terms: the **honest test** is a learner-worded,
> paraphrased test set, and the **easy test** is the course-worded one, the questions as the course wrote them. The lesson
> *Embedding models have blind spots* teaches why a course-worded test flatters a model.
>
> **Word overlap** is how we measure how "borrowed" a question is: the share of its content words, letters only, with the
> small common words dropped (the real term: stop words), that appear in the lesson's Learn text. 1.0 means every word is the lesson's; 0.0 means none is.

### Exercise 4: the overlap of a copy

A learner pastes a whole sentence of the Learn text as their question. What is its word overlap with that lesson?

In [17]:
OVERLAP_OF_COPY = "___"   # a number between 0 and 1

In [18]:
mine = attempt(OVERLAP_OF_COPY, "OVERLAP_OF_COPY")
if mine is not None:
    lesson_text = r.lesson_text(corpus)
    para = corpus[0]
    got = r.word_overlap(para["text"], lesson_text[para["lesson"]])
    assert abs(float(mine) - 1.0) < 0.01 and got == 1.0, "Not quite: every content word of a copied sentence is, by definition, in the lesson."
    print(f"Right: 1.0. Checked on {para['id']}: {got}. The easy test sits nearer that end than a learner ever would.")

Right: 1.0. Checked on use_case_score#0: 1.0. The easy test sits nearer that end than a learner ever would.


Only the **check** and **more** questions of dev and test are rewritten: they are the ones written from the Learn text.
The prepared chips were already written the way a learner asks, and adding them (about 350 more) would triple the time.
First, how borrowed are the originals?

In [19]:
lesson_text = r.lesson_text(corpus)
to_rewrite = [p for p in r.pairs_in(pairs, splits, ("dev", "test")) if p["source"] in ("check", "more")]
orig_overlap = [r.word_overlap(p["question"], lesson_text[p["lesson"]]) for p in to_rewrite]
print(f"{len(to_rewrite)} dev and test questions to rewrite ({dict(Counter(splits[p['lesson']] for p in to_rewrite))}); "
      f"their word overlap with their lesson averages {sum(orig_overlap) / len(orig_overlap):.2f} "
      f"(half of them above {sorted(orig_overlap)[len(orig_overlap) // 2]:.2f})")

211 dev and test questions to rewrite ({'test': 87, 'dev': 124}); their word overlap with their lesson averages 0.55 (half of them above 0.56)


**Your guess first.** After the rewrite, what will the paraphrases' word overlap average? (The originals sit around 0.5.)

In [20]:
you.guess("paraphrase_overlap", None)   # e.g. 0.3

No guess written down. The habit only works if you commit to one before you look.


> **In plain words: how the rewrites are made.** **Ollama** is a program that runs open models on this computer and answers over a
> local web address, so no question leaves the machine. `qwen3:14b`'s 14 billion "learned numbers" are its **parameters** (also
> called weights). Each rewrite runs at **temperature** 0.7 (how varied the wording is: *How an LLM works, in plain words*) with its
> own seed, so a rerun gives the same rewrite, and with the model's thinking turned off so only the one line comes back. Model-made
> test cases need a person's reading, as *Drafted labels, expert-confirmed* says.

**How long this takes.** About 1.2 seconds a question with `qwen3:14b` on this computer after the model loads (about 5 s
the first time), so about **4 to 5 minutes** for the 208 questions. Every rewrite is cached by question id in
`data/paraphrases.json`, so running this cell again costs nothing, and only new questions would be sent to the model.

In [21]:
t0 = time.monotonic()
honest = r.build_honest_test(pairs, splits, ("dev", "test"), corpus=corpus)
paraphrase_seconds = time.monotonic() - t0
print(f"{len(honest)} paraphrases in {paraphrase_seconds / 60:.1f} minutes (0.0 means they were all cached)")
mlflow.log_metrics({"paraphrase_seconds": round(paraphrase_seconds, 1), "n_honest": len(honest)})
mlflow.log_params({"paraphrase_model": r.PARAPHRASE_MODEL, "paraphrase_prompt": r.PARAPHRASE_PROMPT})

honest test: 211 questions in ('dev', 'test'); 208 cached, 3 to paraphrase with qwen3:14b


  3/3 · 6 s
211 paraphrases in 0.1 minutes (0.0 means they were all cached)


In [22]:
mean_orig = sum(h["orig_word_overlap"] for h in honest) / len(honest)
mean_para = sum(h["word_overlap"] for h in honest) / len(honest)
lower = sum(h["word_overlap"] < h["orig_word_overlap"] for h in honest)
you.reveal("paraphrase_overlap", mean_para)
print(f"originals average {mean_orig:.2f}, paraphrases {mean_para:.2f}: a drop of {mean_orig - mean_para:.2f}; "
      f"{lower} of {len(honest)} paraphrases ({lower / len(honest):.0%}) share fewer words than their original")
mlflow.log_metrics({"overlap_original_mean": mean_orig, "overlap_paraphrase_mean": mean_para,
                    "overlap_drop": mean_orig - mean_para, "share_paraphrases_lower_overlap": lower / len(honest)})
show = r.sample_pairs(honest, 5)
pd.DataFrame([{"original": h["orig_question"], "overlap": h["orig_word_overlap"], "paraphrase": h["question"],
               "overlap ": h["word_overlap"]} for h in show])

paraphrase_overlap: it was 37.1% (no guess written down)
originals average 0.55, paraphrases 0.37: a drop of 0.18; 168 of 211 paraphrases (80%) share fewer words than their original


,original,overlap,paraphrase,overlap
0,Agent A succeeds 9 times in 10 at $0.40 per success. Agent B succeeds 8 times in 10 at $0.05 per success. ...,0.364,"If one person makes money 9 out of 10 times at 40 cents each, and another makes money 8 out of 10 times at...",0.000
1,Your Terraform should add one group's grant to a catalog and leave every other grant alone. Which resource...,0.500,"Which Terraform resource can I use to add a group's access to a catalog, but not change anything else?",0.556
2,"Before answering, your main agent retells each helper's report in its own words. The answers read well. Wh...",0.700,"If the main agent just repeats what each helper said in their own words, and the answers still make sense,...",0.625
3,You show people two candidate answers and let them pick the better one. Why show the two in a random order?,0.889,Why would you mix up the order of the two choices when asking someone to pick the better one?,0.429
4,"Predicting which machines will break down would save a lot of downtime. Past failures are on record, but t...",0.250,"If I want to know which machines are likely to break, but the sensor data is split across different system...",0.167


**Reading it.** The paraphrases are a harder test: fewer of their words are the lesson's. *This is wrong if* the model
also changed the **meaning**, so that the right lesson is no longer the right answer. One of the first three rewrites did
exactly that: "Agent A succeeds 9 times in 10" became "one person makes money 9 out of 10 times". A model made these, so
until a person has read them the honest test is **model-made**, and every number scored on it has to say so.

### Your hand check: 20 paraphrases

Twenty paraphrases, picked in proportion to module size (seed 7). For each, decide: does the rewrite still ask the same
thing, so that the original's lesson is still the right answer? Mark `True` (ok) or `False` (not ok) in the next cell.
Unmarked stays `None`. Nothing is written until you mark something, and the answers file fills nothing in here on
purpose: only a person's reading counts.

In [23]:
to_check = r.sample_pairs(honest, 20)
for h in to_check:
    print(f'VERDICTS["{h["id"]}"] = None')
    print(f'    # was : {h["orig_question"][:150]}')
    print(f'    # now : {h["question"][:150]}')

VERDICTS["agent_scorecard?0~0"] = None
    # was : Agent A succeeds 9 times in 10 at $0.40 per success. Agent B succeeds 8 times in 10 at $0.05 per success. Which is better?
    # now : If one person makes money 9 out of 10 times at 40 cents each, and another makes money 8 out of 10 times at 5 cents each, who earns more?
VERDICTS["better_option?0~0"] = None
    # was : A model costs 10 times more per call but resolves 25 more tickets out of every 100, each worth $4. Its calls cost $0.02 instead of $0.002. Is it worth
    # now : If a robot costs more to use each time but handles more problems each time it's used, should we still use it?
VERDICTS["callable_tools?2~0"] = None
    # was : The agent's service principal has USE CATALOG, USE SCHEMA, and EXECUTE on the whole schema `main.support`. A new function is added to it. Can the agen
    # now : If someone has permission to use and run things in a certain part of the database, and a new tool is added there, can they use the new tool?
V

In [24]:
VERDICTS = {h["id"]: None for h in to_check}   # set each to True (same question, same lesson) or False, then run the next cell

In [25]:
state = r.record_checks(VERDICTS)
honest = r.build_honest_test(pairs, splits, ("dev", "test"), corpus=corpus, progress=False)   # cached: free; picks up the flags
honest = [h for h in honest if h["id"] not in set(state["not_ok_ids"])]
mlflow.log_metrics({"paraphrases_checked_by_hand": state["checked"], "paraphrases_checked_by_person": state["checked_by_person"],
                    "paraphrases_agreed": state["agreed"], "paraphrases_not_ok": state["not_ok"], "n_honest_after_checks": len(honest)})
mlflow.set_tag("paraphrases_checked_by", ", ".join(f"{n} by {who}" for who, n in state["by"].items()) or "nobody yet")
if state["checked"] == 0:
    print(f"No paraphrase has been read by a person yet: the honest test ({len(honest)} questions) is model-made until then.")
else:
    who = ", ".join(f"{n} by {by}" for by, n in state["by"].items())
    print(f"{state['checked']} paraphrases read ({who}), {state['agreed']} agreed, {state['not_ok']} dropped from the honest test; "
          f"{state['unmarked']} still unread. The honest test now has {len(honest)} questions.")
    if state["checked_by_person"] < state["checked"]:
        print("A model (Claude) did the reading at the owner's request: the honest test is model-made and model-checked until a person reads it.")

208 paraphrases read (208 by Claude), 197 agreed, 11 dropped from the honest test; 3 still unread. The honest test now has 200 questions.
A model (Claude) did the reading at the owner's request: the honest test is model-made and model-checked until a person reads it.


## 5. Register the datasets in MLflow (the copy that counts)

> **In plain words.** `data/` is a git-ignored working copy. The copy that counts is in MLflow: each dataset is logged with
> its **digest**, a short fingerprint of the rows' text and number columns, so two runs on the same rows show the same
> digest and a run on changed content shows a different one. (MLflow leaves True/False columns out of the digest, which is
> why the honest test's `checked` flag is stored as 0 or 1: your verdicts in section 4 must change the digest, or a read
> test and an unread one would look the same.) Stage 2 and 4 log the same datasets as their inputs, which is how a score
> is tied to the data it was scored on.

Seven datasets: `pairs_train`, `pairs_dev`, `pairs_test`, `pairs_holdout` (registered, rows never shown), `honest_dev`,
`honest_test` and the `corpus`. The rows of every dataset but holdout also go in as JSON artifacts, so they can be read
back from the run.

In [26]:
by_split = {s: r.pairs_in(pairs, splits, s, yes_holdout=(s == r.LOCKED)) for s in r.SPLITS}
honest_by = {s: [h for h in honest if splits[h["lesson"]] == s] for s in ("dev", "test")}
fingerprint = r.content_fingerprint(corpus, pairs)
datasets = {f"pairs_{s}": tracking.table_dataset(by_split[s], f"pairs_{s}", targets="lesson") for s in r.SPLITS}
datasets |= {f"honest_{s}": tracking.table_dataset(honest_by[s], f"honest_{s}", targets="lesson") for s in ("dev", "test")}
datasets["corpus"] = tracking.table_dataset(corpus, "corpus")
context = {"pairs_train": "training", "pairs_dev": "dev", "pairs_test": "test", "pairs_holdout": "holdout (locked)",
           "honest_dev": "dev", "honest_test": "test", "corpus": "corpus"}
for name, ds in datasets.items():
    mlflow.log_input(ds, context=context[name])
    if name != "pairs_holdout":
        mlflow.log_dict(by_split[name[6:]] if name.startswith("pairs_") else honest_by[name[7:]] if name.startswith("honest_") else corpus, f"{name}.json")
    print(f"{name:14} {len(ds.df):5} rows · digest {ds.digest}")
mlflow.log_params({"content_fingerprint": fingerprint, "k": r.K})
mlflow.log_artifact(str(r.DATA / "splits.json"))
mlflow.log_dict(ambiguous, "ambiguous_questions.json")
mlflow.log_dict({"pairs_by_source": dict(by_source), "lessons_by_split": counts, "pairs_by_split": dict(pairs_by_split),
                 "honest_by_split": {s: len(v) for s, v in honest_by.items()}, "prepared_chips_by_kind": dict(chips)}, "counts.json")
tracking.save_json("pairs.json", [{**p, "split": splits[p["lesson"]]} for p in pairs if splits[p["lesson"]] != r.LOCKED])   # local working copy without the holdout rows (git-ignored); holdout lives in MLflow by digest only
tracking.save_json("honest.json", honest)
print(f"content fingerprint {fingerprint}; working copies in {r.DATA}")

pairs_train      970 rows · digest 720bbfb9


pairs_dev        321 rows · digest 8f1cabad
pairs_test       236 rows · digest 4fbb10ae
pairs_holdout    153 rows · digest e88c397c
honest_dev       119 rows · digest 34f8f882
honest_test       81 rows · digest 6580a642


corpus           744 rows · digest 527a1c48


content fingerprint a49814aad9cf; working copies in /Users/huydo/Developer/dd-retrieval/notebooks/learning/coach_retrieval/data


## 6. What this settles, and the description that goes with the run

The run's description says what was built, why, how, the counts and the caveats, so someone opening it in a year (or on
Databricks, where the same datasets become Delta tables) does not need this notebook to understand it.

In [27]:
description = f'''**Stage 1 of the coach-retrieval lab: the pairs, the frozen split and the honest test.** Built by `01_pairs.ipynb`; the datasets logged here are the copy that counts (digests above). Content fingerprint {fingerprint}.

**What.** {len(corpus)} Learn paragraphs (the corpus, always searched whole) from {len(data['lessons'])} lessons in {len(data['modules'])} modules; {len(pairs)} pairs (a question with its one known right lesson): check {by_source['check']}, more {by_source['more']}, prepared chips {by_source['prepared']} ({dict(chips)}). {len(ambiguous)} question texts ({dropped_copies} copies) dropped for appearing under several lessons. Honest test: {len(honest)} learner-worded rewrites of the dev and test check and more questions ({ {s: len(v) for s, v in honest_by.items()} }), made by {r.PARAPHRASE_MODEL} through Ollama in {paraphrase_seconds / 60:.1f} minutes this pass (cached in data/paraphrases.json).

**Why.** Today's coach does not retrieve with embeddings (the page tells it the lesson), so the champion for the retrieval step is bge-large as shipped (recall@5 0.84 on Check questions, PR #170), and a fine-tune has to earn its place where the lesson is not known. That needs a test the fine-tune cannot have seen (held-out lessons) and questions that do not borrow the lesson's words (the honest test).

**How.** Split by whole lesson, within each module, seeded hash order (seed {r.SEED}), 60/20/10/10: lessons {counts}; pairs {dict((s, pairs_by_split[s]) for s in r.SPLITS)}. Frozen in data/splits.json (refuses to redeal). Leakage checks: {sum(c['passed'] for c in checks)} of {len(checks)} passed ({'; '.join(c['check'] + ': ' + ('ok' if c['passed'] else 'LOOK') for c in checks)}). Holdout is registered with its digest only; its rows are never shown and scoring it needs yes_holdout=True, once, at the end.

**Numbers.** Word overlap with the lesson: originals {mean_orig:.2f}, paraphrases {mean_para:.2f} (drop {mean_orig - mean_para:.2f}; {lower / len(honest):.0%} of rewrites share fewer words). Paraphrases read by a person: {state['checked']} ({state['agreed']} agreed, {state['not_ok']} dropped).

**Caveats.** (1) The honest test is model-made until a person has read it; one of the first three rewrites changed the meaning. (2) Option chips ("why not <wrong answer>?") name no topic: bge-large finds their lesson 0.45 of the time (174 questions, Wilson range 0.38 to 0.53) against 0.84 to 0.88 (ranges 0.75 to 0.95) for check and more questions (smoke run, 328 dev questions); score them as their own group or leave them out and say why. (3) Check and more questions share words with the Learn text, so a word-matching rule looks good on them; the honest test is the fair one. (4) Three modules have fewer than four lessons; their per-module numbers rest on a handful of questions. (5) No real typed learner questions exist on this computer yet (0 events); when they arrive they join the honest test.'''
mlflow.set_tag("mlflow.note.content", description)
print(description)

**Stage 1 of the coach-retrieval lab: the pairs, the frozen split and the honest test.** Built by `01_pairs.ipynb`; the datasets logged here are the copy that counts (digests above). Content fingerprint a49814aad9cf.

**What.** 744 Learn paragraphs (the corpus, always searched whole) from 212 lessons in 28 modules; 1680 pairs (a question with its one known right lesson): check 212, more 418, prepared chips 1050 ({'term': 129, 'option': 888, 'explain:gotcha': 33}). 10 question texts (329 copies) dropped for appearing under several lessons. Honest test: 200 learner-worded rewrites of the dev and test check and more questions ({'dev': 119, 'test': 81}), made by qwen3:14b through Ollama in 0.1 minutes this pass (cached in data/paraphrases.json).

**Why.** Today's coach does not retrieve with embeddings (the page tells it the lesson), so the champion for the retrieval step is bge-large as shipped (recall@5 0.84 on Check questions, PR #170), and a fine-tune has to earn its place where the le

In [28]:
mlflow.end_run()
print(f"This pass is saved: {tracking.run_link(run)}")

This pass is saved: http://localhost:5050/#/experiments/6 · run "pairs · build and freeze (seed 7)" (b5d212a7)


**What comes next.** `02_baselines.ipynb` scores the champion (`bge-large` as shipped), `embeddinggemma` and the
word-matching rule on `pairs_dev` and `honest_dev`, with Wilson ranges and a paired test, and sets the bar a fine-tune has
to clear. It reads the datasets registered here by name and digest, never from a fresh deal.